In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "bohn2016role")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Bohn_2016_roleofpastinteractions_data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:


import pandas as pd
import numpy as np
df = pd.read_csv(complete_path_1)


df['study_id']="bohn2016role"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)

df = df.rename(columns={"test trial": "trial", 
        "subject": "ape"})

In [3]:
df['ape'] = df['ape'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)  
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

df = df.rename(columns={"species_y": "species",
    "sex_y": "sex", 
    "food left": "food_left",
    "food right": "food_right",
    "side indicated": "side_indicated",
    "food indicated": "food_indicated",
    "point to empty": "point_to_empty"})


In [4]:
df.rename(columns={"ape": "participant"}, inplace=True)
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df= df.merge(ape_dob,left_on='participant', right_on='name', how='left') #insert dob of participants
df['dodc'] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
df['dodc'] = pd.to_datetime(df['dodc'])##convert date of data collection to datetime format
df['dob'] = pd.to_datetime(df['dob'])##convert date of birth to datetime format

df['age_in_years'] = (df['dodc'] - df['dob']).dt.days//365

In [5]:
# df.columns

In [6]:
bohn2016role_standardized =df[['study_id','year', 'month', 'day', 'participant', 'age_in_years',
        'sex','species',  'phase', 'session', 'trial','see',
       'bring', 'condition',  'food_left', 'food_right',
       'point', 'side_indicated', 'food_indicated', 'point_to_empty']]

In [7]:
comp_out_path_stand = os.path.join(out_pathway, 'bohn2016role_standardized.csv')
bohn2016role_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


In [8]:
names =bohn2016role_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
bohn2016role_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'bohn2016role_glossary.csv')
bohn2016role_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
